# UMAP unificado: clases, origen y porcentaje de las fracciones

Basado en `15.Umap_unificado.ipynb`. Se reutiliza su CSV con metadata y coordenadas para mostrar **la misma proyección** de tres maneras:

1. Por clase, como en el notebook original, incluyendo imágenes completas y `sin_clase`.
2. Con dos colores: **tope (borde)** y **base**, excluyendo los perfiles completos.
3. Con una escala continua de azules según `porcentaje_avance`: **100 % = azul oscuro**; las fracciones menores tienen tonos progresivamente más claros. Incluye las imágenes completas.

El porcentaje representa la proporción del perfil conservada en el recorte, no un porcentaje de área o de píxeles. Un 50 % desde el tope y un 50 % desde la base reciben el mismo tono.

UMAP se ajustó sobre todas las imágenes sin usar etiquetas. Aquí solo cambia la visualización: no se vuelve a ajustar al filtrar las completas. Es una exploración del conjunto unificado, que incluye imágenes de entrenamiento; no mide generalización.

**Requisito:** ejecutar el notebook 17 si el CSV indicado en la configuración todavía no existe o si se desea actualizar el modelo, los datos o los parámetros de UMAP.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from IPython.display import display

from vasijas.config import load_config
from vasijas.paths import ProjectPaths, ruta_relativa
from vasijas.umap_evaluacion import graficar_umap_test

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)
MODELO = "model_resnet18_unificado_bs16_lr1e-3_weighted_unfreeze"
CSV_PATH = PATHS.OUTPUT_UMAP / f"umap_unificado_{MODELO}.csv"
OUTPUT_DIR = PATHS.OUTPUT_UMAP
FIGURE_PATHS = {
    "clases": OUTPUT_DIR / f"umap_unificado_clases_{MODELO}.png",
    "origen": OUTPUT_DIR / f"umap_unificado_origen_fracciones_{MODELO}.png",
    "porcentaje": OUTPUT_DIR / f"umap_unificado_porcentaje_{MODELO}.png",
}
print(f"Proyección de origen: {ruta_relativa(CSV_PATH)}")

## Lectura y verificación

Se comprueban las coordenadas, la identidad y los metadatos usados para colorear. Las únicas categorías admitidas son `completo`, `tope` y `base`. Los perfiles completos deben indicar 100 % y los recortes un porcentaje mayor que 0 y menor que 100. Cualquier valor faltante o inesperado detiene el proceso para evitar excluir puntos silenciosamente.

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"No existe {CSV_PATH}. Ejecutá primero 17.Umap_unificado.ipynb."
    )
tabla_umap = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
requeridas = [
    "Identificador",
    "ruta_relativa",
    "clase_carpeta",
    "borde_origen",
    "porcentaje_avance",
    "UMAP_1",
    "UMAP_2",
]
faltantes = set(requeridas) - set(tabla_umap.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en el CSV: {sorted(faltantes)}")
if tabla_umap.empty or tabla_umap[requeridas].isna().any().any():
    raise ValueError("El CSV debe contener imágenes y metadata completa para colorear.")
if not all(tabla_umap[c].is_unique for c in ["Identificador", "ruta_relativa"]):
    raise ValueError("Debe existir exactamente un punto por imagen.")
if not np.isfinite(tabla_umap[["UMAP_1", "UMAP_2"]].to_numpy(dtype=float)).all():
    raise ValueError("Las coordenadas deben ser finitas.")

origen = tabla_umap["borde_origen"].astype("string").str.strip().str.lower()
inesperados = set(origen) - {"completo", "tope", "base"}
if inesperados:
    raise ValueError(f"Orígenes desconocidos: {sorted(inesperados)}")
porcentaje = pd.to_numeric(tabla_umap["porcentaje_avance"], errors="raise")
es_completa = origen.eq("completo")
es_fraccion = origen.isin(["tope", "base"])
if not np.isfinite(porcentaje.to_numpy(dtype=float)).all():
    raise ValueError("Los porcentajes deben ser finitos.")
if not porcentaje.loc[es_completa].eq(100).all():
    raise ValueError("Las imágenes completas deben indicar 100 %.")
if not porcentaje.loc[es_fraccion].between(0, 100, inclusive="neither").all():
    raise ValueError(
        "Los recortes deben indicar un porcentaje entre 0 y 100 exclusivo."
    )
if not {"tope", "base"}.issubset(set(origen)):
    raise ValueError("Se necesitan fracciones de tope y de base para comparar.")

fracciones = tabla_umap.loc[es_fraccion].copy()
display(
    pd.DataFrame({"origen": origen, "porcentaje": porcentaje})
    .groupby("origen")
    .agg(
        imagenes=("porcentaje", "size"),
        porcentaje_minimo=("porcentaje", "min"),
        porcentaje_maximo=("porcentaje", "max"),
    )
)
print(
    f"Total: {len(tabla_umap)} · Fracciones: {len(fracciones)} "
    f"· Completas excluidas del segundo gráfico: {int(es_completa.sum())}"
)

## 1. Gráfico original por clase

Incluye todas las imágenes. Los tres gráficos mantienen los mismos límites y coordenadas para facilitar la comparación.

In [ ]:
figura_clases = graficar_umap_test(tabla_umap)
eje_clases = figura_clases.axes[0]
eje_clases.set_title(
    f"UMAP de características ResNet18 · unificado (n={len(tabla_umap)})"
)
limites_x = eje_clases.get_xlim()
limites_y = eje_clases.get_ylim()
display(figura_clases)
plt.close(figura_clases)

## 2. Fracciones por origen: tope y base

Solo se representan los recortes: **tope (borde) en naranja** y **base en azul**. Las imágenes completas se excluyen únicamente de esta vista. La leyenda indica cuántas fracciones pertenecen a cada grupo.

In [ ]:
figura_origen, eje_origen = plt.subplots(figsize=(11, 8))
for categoria, etiqueta, color in [
    ("tope", "Tope (borde)", "#E69F00"),
    ("base", "Base", "#0072B2"),
]:
    grupo = tabla_umap.loc[origen.eq(categoria)]
    eje_origen.scatter(
        grupo["UMAP_1"],
        grupo["UMAP_2"],
        color=color,
        s=14,
        alpha=0.65,
        linewidths=0,
        label=f"{etiqueta} (n={len(grupo)})",
    )
eje_origen.set(
    title=f"UMAP unificado · fracciones por origen (n={len(fracciones)})",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    xlim=limites_x,
    ylim=limites_y,
)
eje_origen.legend(
    title="Origen del recorte", bbox_to_anchor=(1.02, 1), loc="upper left"
)
figura_origen.tight_layout()
display(figura_origen)
plt.close(figura_origen)

## 3. Porcentaje conservado: escala de azules

La escala se fija entre 0 y 100 %. Se usa el porcentaje directamente, sin invertirlo: cuanto más perfil conserva la imagen, más oscuro es el azul. Los perfiles completos tienen el tono más oscuro. Se dibujan primero los porcentajes menores para que las imágenes completas queden visibles donde se superponen puntos.

In [ ]:
figura_porcentaje, eje_porcentaje = plt.subplots(figsize=(11, 8))
orden = np.argsort(porcentaje.to_numpy(), kind="stable")
puntos = eje_porcentaje.scatter(
    tabla_umap.iloc[orden]["UMAP_1"],
    tabla_umap.iloc[orden]["UMAP_2"],
    c=porcentaje.iloc[orden],
    cmap="Blues",
    norm=Normalize(vmin=0, vmax=100),
    s=14,
    alpha=1.0,
    linewidths=0,
)
barra = figura_porcentaje.colorbar(puntos, ax=eje_porcentaje)
barra.set_label("Perfil conservado (%) · 100 % = imagen completa")
barra.set_ticks([0, 25, 50, 75, 100])
barra.set_ticklabels(["0 %", "25 %", "50 %", "75 %", "100 %"])
eje_porcentaje.set(
    title=f"UMAP unificado · porcentaje conservado (n={len(tabla_umap)})",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
    xlim=limites_x,
    ylim=limites_y,
)
figura_porcentaje.tight_layout()
display(figura_porcentaje)
plt.close(figura_porcentaje)

## Exportación

Se guardan las tres figuras en PNG a 200 dpi en `../../output/umap`, con nombres propios de estas vistas. Las coordenadas y la metadata proceden del CSV generado por el notebook 17.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for nombre, figura in {
    "clases": figura_clases,
    "origen": figura_origen,
    "porcentaje": figura_porcentaje,
}.items():
    figura.savefig(FIGURE_PATHS[nombre], dpi=200, bbox_inches="tight")
    print(f"Figura guardada: {ruta_relativa(FIGURE_PATHS[nombre])}")